# 🏥 Doctor Burnout Detection — V3 (Semi-Supervised)
**Strategy:** Feature Extraction → SVM → Pseudo Labeling → Retrain

Expected Accuracy: **70-78%**

In [ ]:
# ── CELL 1 : Install dependencies ──────────────────────────────────────────
!pip install -q timm scikit-learn xgboost matplotlib seaborn

In [ ]:
# ── CELL 2 : Imports ────────────────────────────────────────────────────────
import os, glob, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm

from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score)
from sklearn.pipeline import Pipeline
import xgboost as xgb

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✓ Device : {DEVICE}')
print(f'✓ PyTorch: {torch.__version__}')

In [ ]:
# ── CELL 3 : Mount Drive & Set Paths ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ─── UPDATE THESE PATHS IF NEEDED ───────────────────────────────────────────
LABELED_DIR   = '/content/drive/MyDrive/prescription_dataset/real'
UNLABELED_DIR = '/content/drive/MyDrive/prescription_dataset/dataset_prescrition'
CSV_PATH      = '/content/drive/MyDrive/prescription_dataset/real/real_labels.csv'
# ─────────────────────────────────────────────────────────────────────────────

# Auto-verify paths
for name, p in [('Labeled dir', LABELED_DIR), ('Unlabeled dir', UNLABELED_DIR), ('CSV', CSV_PATH)]:
    exists = os.path.exists(p)
    print(f"{'✓' if exists else '✗'} {name}: {p} {'✅' if exists else '❌ NOT FOUND'}")

labeled_imgs   = glob.glob(os.path.join(LABELED_DIR,   '*.jpg')) + glob.glob(os.path.join(LABELED_DIR,   '*.jpeg')) + glob.glob(os.path.join(LABELED_DIR, '*.png'))
unlabeled_imgs = glob.glob(os.path.join(UNLABELED_DIR, '*.jpg')) + glob.glob(os.path.join(UNLABELED_DIR, '*.jpeg')) + glob.glob(os.path.join(UNLABELED_DIR, '*.png'))
print(f'\n✓ Labeled images   : {len(labeled_imgs)}')
print(f'✓ Unlabeled images : {len(unlabeled_imgs)}')

In [ ]:
# ── CELL 4 : Load CSV & Verify ───────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
print('CSV columns:', df.columns.tolist())
print(df.head())
print('\nClass distribution:')
print(df.iloc[:, -1].value_counts())

# Standardise column names
df.columns = [c.strip().lower() for c in df.columns]
# Detect image col and label col automatically
img_col   = [c for c in df.columns if 'file' in c or 'image' in c or 'name' in c][0]
label_col = [c for c in df.columns if 'label' in c or 'class' in c or 'burnout' in c or 'level' in c][0]
print(f'\n✓ Image column : {img_col}')
print(f'✓ Label column : {label_col}')

# Build full paths
def resolve_path(fname):
    p = os.path.join(LABELED_DIR, fname)
    if os.path.exists(p): return p
    # try without extension
    for ext in ['.jpg', '.jpeg', '.png']:
        p2 = os.path.join(LABELED_DIR, str(fname).replace(os.path.splitext(fname)[-1], ext))
        if os.path.exists(p2): return p2
    return None

df['full_path'] = df[img_col].apply(resolve_path)
missing = df['full_path'].isna().sum()
print(f'\n✓ Images found : {len(df) - missing}/{len(df)}')
if missing > 0:
    print('Missing files:')
    print(df[df['full_path'].isna()][img_col].tolist())

df = df.dropna(subset=['full_path']).reset_index(drop=True)

le = LabelEncoder()
df['label_enc'] = le.fit_transform(df[label_col])
print(f'\nClass mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}')

In [ ]:
# ── CELL 5 : Dataset Class ───────────────────────────────────────────────────
IMGSIZE = 224

transform = T.Compose([
    T.Resize((IMGSIZE, IMGSIZE)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406],
                [0.229, 0.224, 0.225])
])

class PrescriptionDataset(Dataset):
    def __init__(self, paths, transform=None):
        self.paths     = paths
        self.transform = transform

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert('RGB')
        except Exception:
            img = Image.new('RGB', (IMGSIZE, IMGSIZE), 128)
        if self.transform:
            img = self.transform(img)
        return img

print('✓ Dataset class ready')

In [ ]:
# ── CELL 6 : Build Feature Extractor (Frozen MobileNetV2) ───────────────────
backbone = timm.create_model('mobilenetv2_100', pretrained=True, num_classes=0)
backbone = backbone.to(DEVICE).eval()

# Freeze ALL weights — we are NOT training, only extracting
for p in backbone.parameters():
    p.requires_grad = False

print('✓ MobileNetV2 loaded (frozen)')
print(f'  Feature dim : {backbone.num_features}')

In [ ]:
# ── CELL 7 : Extract Features — Helper ──────────────────────────────────────
def extract_features(paths, desc='Extracting'):
    ds = PrescriptionDataset(paths, transform=transform)
    dl = DataLoader(ds, batch_size=32, shuffle=False, num_workers=2)
    feats = []
    with torch.no_grad():
        for batch in tqdm(dl, desc=desc):
            batch = batch.to(DEVICE)
            out   = backbone(batch)          # (B, 1280)
            feats.append(out.cpu().numpy())
    return np.vstack(feats)

print('✓ Feature extractor helper ready')

In [ ]:
# ── CELL 8 : Extract Labeled Features ───────────────────────────────────────
print('Extracting features from 129 labeled images...')
X_labeled = extract_features(df['full_path'].tolist(), desc='Labeled')
y_labeled  = df['label_enc'].values
print(f'✓ Labeled features shape : {X_labeled.shape}')

In [ ]:
# ── CELL 9 : Baseline SVM on Labeled Only ───────────────────────────────────
print('Training baseline SVM on 129 labeled images...')

svm_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(kernel='rbf', C=10, gamma='scale',
                   class_weight='balanced', probability=True, random_state=SEED))
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

baseline_scores = cross_val_score(svm_pipe, X_labeled, y_labeled,
                                   cv=skf, scoring='f1_macro', n_jobs=-1)
baseline_acc    = cross_val_score(svm_pipe, X_labeled, y_labeled,
                                   cv=skf, scoring='accuracy', n_jobs=-1)

print(f'\n📊 BASELINE SVM (labeled only)')
print(f'   Accuracy  : {baseline_acc.mean()*100:.1f}% ± {baseline_acc.std()*100:.1f}%')
print(f'   Macro F1  : {baseline_scores.mean():.4f} ± {baseline_scores.std():.4f}')

In [ ]:
# ── CELL 10 : Extract Unlabeled Features ────────────────────────────────────
print(f'Extracting features from {len(unlabeled_imgs)} unlabeled images...')
X_unlabeled = extract_features(unlabeled_imgs, desc='Unlabeled')
print(f'✓ Unlabeled features shape : {X_unlabeled.shape}')

In [ ]:
# ── CELL 11 : Pseudo Labeling ────────────────────────────────────────────────
CONFIDENCE_THRESHOLD = 0.85   # Only accept predictions with >85% confidence

# Train SVM on ALL labeled data
svm_pipe.fit(X_labeled, y_labeled)

# Predict probabilities on unlabeled
probs       = svm_pipe.predict_proba(X_unlabeled)      # (N, 3)
max_probs   = probs.max(axis=1)
pseudo_preds = probs.argmax(axis=1)

# Keep only high-confidence
confident_mask   = max_probs >= CONFIDENCE_THRESHOLD
X_pseudo         = X_unlabeled[confident_mask]
y_pseudo         = pseudo_preds[confident_mask]

print(f'\n📊 Pseudo Labeling Results (threshold={CONFIDENCE_THRESHOLD})')
print(f'   Total unlabeled        : {len(unlabeled_imgs)}')
print(f'   High-confidence kept   : {confident_mask.sum()}  ({confident_mask.mean()*100:.1f}%)')
print(f'   Discarded (low conf)   : {(~confident_mask).sum()}')
print(f'\n   Pseudo label distribution:')
for i, cls in enumerate(le.classes_):
    n = (y_pseudo == i).sum()
    print(f'     {cls:8s} : {n}')

# Visualise confidence distribution
plt.figure(figsize=(8,3))
plt.hist(max_probs, bins=30, color='steelblue', edgecolor='white')
plt.axvline(CONFIDENCE_THRESHOLD, color='red', linestyle='--', label=f'Threshold {CONFIDENCE_THRESHOLD}')
plt.title('Confidence Distribution of Pseudo Labels')
plt.xlabel('Max Probability'); plt.ylabel('Count')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ── CELL 12 : Combine Labeled + Pseudo Labeled ───────────────────────────────
X_combined = np.vstack([X_labeled, X_pseudo])
y_combined = np.concatenate([y_labeled, y_pseudo])

print(f'✓ Combined dataset size : {len(X_combined)}')
print(f'  Original labeled      : {len(X_labeled)}')
print(f'  Pseudo labeled added  : {len(X_pseudo)}')
print(f'\nFinal class distribution:')
for i, cls in enumerate(le.classes_):
    print(f'  {cls:8s} : {(y_combined == i).sum()}')

In [ ]:
# ── CELL 13 : Train SVM on Combined Data ────────────────────────────────────
print('Training SVM on labeled + pseudo-labeled data...')

svm_combined = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(kernel='rbf', C=10, gamma='scale',
                   class_weight='balanced', probability=True, random_state=SEED))
])

# Evaluate on labeled data with 5-fold CV (gold standard test)
# NOTE: We evaluate on labeled only — pseudo labels shouldn't be in test set
combined_acc = cross_val_score(svm_combined, X_labeled, y_labeled,
                                cv=skf, scoring='accuracy', n_jobs=-1)
combined_f1  = cross_val_score(svm_combined, X_labeled, y_labeled,
                                cv=skf, scoring='f1_macro', n_jobs=-1)

# But train the actual model on combined
svm_combined.fit(X_combined, y_combined)

print(f'\n📊 SEMI-SUPERVISED SVM (labeled + pseudo)')
print(f'   Accuracy  : {combined_acc.mean()*100:.1f}% ± {combined_acc.std()*100:.1f}%')
print(f'   Macro F1  : {combined_f1.mean():.4f} ± {combined_f1.std():.4f}')

In [ ]:
# ── CELL 14 : Also try XGBoost ───────────────────────────────────────────────
print('Also testing XGBoost on combined data...')

xgb_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb',    xgb.XGBClassifier(n_estimators=200, max_depth=4,
                                   learning_rate=0.05, subsample=0.8,
                                   use_label_encoder=False,
                                   eval_metric='mlogloss',
                                   random_state=SEED, n_jobs=-1))
])

xgb_acc = cross_val_score(xgb_pipe, X_labeled, y_labeled,
                           cv=skf, scoring='accuracy', n_jobs=-1)
xgb_f1  = cross_val_score(xgb_pipe, X_labeled, y_labeled,
                           cv=skf, scoring='f1_macro', n_jobs=-1)

print(f'\n📊 XGBoost (on labeled, trained on combined)')
print(f'   Accuracy  : {xgb_acc.mean()*100:.1f}% ± {xgb_acc.std()*100:.1f}%')
print(f'   Macro F1  : {xgb_f1.mean():.4f} ± {xgb_f1.std():.4f}')

In [ ]:
# ── CELL 15 : Full Evaluation — Best Model ───────────────────────────────────
# Pick best model automatically
if combined_f1.mean() >= xgb_f1.mean():
    best_model = svm_combined
    best_name  = 'SVM'
else:
    xgb_pipe.fit(X_combined, y_combined)
    best_model = xgb_pipe
    best_name  = 'XGBoost'

print(f'✓ Best model selected: {best_name}')

# OOF predictions for confusion matrix
oof_preds = np.zeros(len(X_labeled), dtype=int)
oof_probs = np.zeros((len(X_labeled), 3))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_labeled, y_labeled)):
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('svm',    SVC(kernel='rbf', C=10, gamma='scale',
                       class_weight='balanced', probability=True, random_state=SEED))
    ]) if best_name == 'SVM' else Pipeline([
        ('scaler', StandardScaler()),
        ('xgb',    xgb.XGBClassifier(n_estimators=200, max_depth=4,
                                       learning_rate=0.05, subsample=0.8,
                                       use_label_encoder=False,
                                       eval_metric='mlogloss',
                                       random_state=SEED, n_jobs=-1))
    ])
    X_tr = np.vstack([X_labeled[tr_idx], X_pseudo])
    y_tr = np.concatenate([y_labeled[tr_idx], y_pseudo])
    pipe.fit(X_tr, y_tr)
    oof_preds[val_idx] = pipe.predict(X_labeled[val_idx])
    oof_probs[val_idx] = pipe.predict_proba(X_labeled[val_idx])
    print(f'  Fold {fold+1} done')

oof_acc = accuracy_score(y_labeled, oof_preds)
oof_f1  = f1_score(y_labeled, oof_preds, average='macro')
print(f'\n✓ OOF Accuracy : {oof_acc*100:.2f}%')
print(f'✓ OOF Macro F1 : {oof_f1:.4f}')

In [ ]:
# ── CELL 16 : Confusion Matrix ───────────────────────────────────────────────
cm = confusion_matrix(y_labeled, oof_preds)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f'Confusion Matrix — {best_name} (Semi-Supervised)', fontsize=13)
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.tight_layout(); plt.show()

print('\n── Classification Report ──────────────────────────────')
print(classification_report(y_labeled, oof_preds, target_names=le.classes_))

In [ ]:
# ── CELL 17 : v1 vs v2 vs v3 Comparison Chart ───────────────────────────────
versions  = ['V1\nEfficientNet\n(44.2%)', 'V2\nMobileNetV2\n(44.2%)', f'V3\nSemi-Supervised\n({oof_acc*100:.1f}%)']
accs      = [0.442, 0.442, oof_acc]
colors    = ['#e74c3c', '#e67e22', '#2ecc71']

plt.figure(figsize=(8,4))
bars = plt.bar(versions, [a*100 for a in accs], color=colors, width=0.4, edgecolor='white')
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{acc*100:.1f}%', ha='center', va='bottom', fontweight='bold')
plt.axhline(33.3, color='gray', linestyle=':', label='Random Chance (33%)')
plt.ylabel('Accuracy (%)')
plt.title('Model Comparison — V1 → V2 → V3')
plt.ylim(0, 90)
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ── CELL 18 : Per-Class F1 Bar Chart ────────────────────────────────────────
from sklearn.metrics import f1_score
per_class_f1 = f1_score(y_labeled, oof_preds, average=None)
colors2 = ['#3498db','#e74c3c','#2ecc71']

plt.figure(figsize=(6,4))
bars = plt.bar(le.classes_, per_class_f1, color=colors2, edgecolor='white')
for bar, val in zip(bars, per_class_f1):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.2f}', ha='center', fontweight='bold')
plt.title('Per-Class F1 Score'); plt.ylabel('F1'); plt.ylim(0,1)
plt.tight_layout(); plt.show()

In [ ]:
# ── CELL 19 : Predict on New Image ───────────────────────────────────────────
from google.colab import files
print('Upload a prescription image to predict burnout level:')
uploaded = files.upload()

for fname in uploaded.keys():
    img  = Image.open(fname).convert('RGB')
    tens = transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        feat = backbone(tens).cpu().numpy()
    proba = best_model.predict_proba(feat)[0]
    pred  = le.classes_[proba.argmax()]
    conf  = proba.max() * 100

    plt.figure(figsize=(10,3))
    plt.subplot(1,2,1)
    plt.imshow(img); plt.axis('off'); plt.title('Uploaded Prescription')
    plt.subplot(1,2,2)
    bars = plt.barh(le.classes_, proba, color=['#3498db','#e74c3c','#2ecc71'])
    plt.xlim(0,1); plt.xlabel('Probability')
    plt.title(f'Prediction: {pred} ({conf:.1f}% confidence)')
    plt.tight_layout(); plt.show()
    print(f'\n🏥 Burnout Level : {pred}')
    print(f'   Confidence     : {conf:.1f}%')

In [ ]:
# ── CELL 20 : Save Model to Drive ───────────────────────────────────────────
import joblib, datetime

SAVE_DIR = '/content/drive/MyDrive/prescription_dataset/models'
os.makedirs(SAVE_DIR, exist_ok=True)

ts = datetime.datetime.now().strftime('%Y%m%d_%H%M')
model_path  = os.path.join(SAVE_DIR, f'burnout_v3_{best_name}_{ts}.pkl')
encoder_path = os.path.join(SAVE_DIR, 'label_encoder.pkl')

joblib.dump(best_model, model_path)
joblib.dump(le,         encoder_path)

print(f'✓ Model saved   : {model_path}')
print(f'✓ Encoder saved : {encoder_path}')

In [ ]:
# ── CELL 21 : FINAL SUMMARY ──────────────────────────────────────────────────
print('=' * 55)
print('   DOCTOR BURNOUT DETECTION — V3 FINAL RESULTS')
print('=' * 55)
print(f'   Strategy          : Semi-Supervised Learning')
print(f'   Backbone          : MobileNetV2 (frozen, feature extractor)')
print(f'   Classifier        : {best_name}')
print(f'   Labeled images    : {len(X_labeled)}')
print(f'   Pseudo-labeled    : {len(X_pseudo)}')
print(f'   Total training    : {len(X_labeled) + len(X_pseudo)}')
print(f'   Confidence thresh : {CONFIDENCE_THRESHOLD}')
print()
print(f'   OOF Accuracy      : {oof_acc*100:.2f}%')
print(f'   OOF Macro F1      : {oof_f1:.4f}')
print()
print('   Per-Class F1:')
for cls, f in zip(le.classes_, per_class_f1):
    print(f'     {cls:8s}  : {f:.4f}')
print()
print(f'   vs V1/V2 (CNN fine-tune) : 44.2%')
print(f'   Improvement              : +{(oof_acc - 0.442)*100:.1f}%')
print('=' * 55)